In [ ]:
"""
collect_legit_urls.py
=====================
Thu thập Legit URLs có path — KHÔNG cần kết nối server nước ngoài.

Chiến lược:
  1. Download Tranco list (1 file CSV ~10MB, CDN nhanh)
  2. Với mỗi domain trong Tranco → fetch sitemap.xml trực tiếp
     (kết nối thẳng đến domain đích, không qua proxy trung gian)
  3. Parse sitemap bằng regex thuần, xử lý sitemap index

Cài đặt:
  pip install requests tqdm
"""

import csv
import logging
import random
import re
import time
from collections import defaultdict
from io import StringIO
from pathlib import Path
from urllib.parse import urlparse

import requests
from tqdm import tqdm

# ── Cấu hình ────────────────────────────────────────────────────────────────

OUTPUT_FILE    = "legit_urls.csv"
LOG_FILE       = "collect.log"
TARGET_TOTAL   = 25000
MAX_PER_DOMAIN = 50
HTTP_TIMEOUT   = 10
SLEEP_OK       = 0.5
SLEEP_ERR      = 1.0

# Số domain Tranco thử (tăng nếu muốn nhiều URL hơn)
TRANCO_TOP_N   = 700

SKIP_EXT = {
    ".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp", ".ico",
    ".pdf", ".zip", ".rar", ".gz", ".tar",
    ".xml", ".css", ".js", ".json",
    ".mp4", ".mp3", ".avi", ".mov",
    ".woff", ".woff2", ".ttf", ".eot",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# Domain ưu tiên thêm vào đầu danh sách Tranco
PRIORITY_DOMAINS = [
    "vnexpress.net", "tuoitre.vn", "thanhnien.vn", "dantri.com.vn",
    "zingnews.vn", "vietnamnet.vn", "tienphong.vn", "laodong.vn",
    "kenh14.vn", "24h.com.vn", "baomoi.com", "nhandan.vn",
    "shopee.vn", "lazada.vn", "tiki.vn", "thegioididong.com",
    "dienmayxanh.com", "bachhoaxanh.com", "sendo.vn",
    "chinhphu.vn", "hust.edu.vn",
    "bbc.com", "reuters.com", "theguardian.com",
    "github.com", "stackoverflow.com", "wikipedia.org",
    "medium.com", "dev.to", "geeksforgeeks.org",
    "amazon.com", "ebay.com", "reddit.com",
]

# ── Logging ──────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)

# ── Helpers ──────────────────────────────────────────────────────────────────

def is_valid_url(url: str) -> bool:
    try:
        p = urlparse(url.strip())
    except Exception:
        return False
    if p.scheme not in ("http", "https"):
        return False
    if not p.netloc:
        return False
    if not p.path or p.path == "/":
        return False
    if Path(p.path).suffix.lower() in SKIP_EXT:
        return False
    return True


def domain_of(url: str) -> str:
    return urlparse(url).netloc.lower()


def netloc_matches(netloc: str, domain: str) -> bool:
    return netloc == domain or netloc.endswith("." + domain)


# ── Tranco list ───────────────────────────────────────────────────────────────

def load_tranco_domains(top_n: int = TRANCO_TOP_N) -> list:
    """
    Download Tranco top-1M list (file CSV, ~10MB).
    URL trực tiếp từ Tranco CDN — thường không bị block.
    Cache vào file local để tránh download lại.
    """
    cache_file = Path(".tranco_cache.csv")

    if cache_file.exists():
        log.info("Dùng Tranco cache: %s", cache_file)
        text = cache_file.read_text(encoding="utf-8")
    else:
        log.info("Đang download Tranco list...")
        # URL stable, luôn trỏ đến list mới nhất
        url = "https://tranco-list.eu/top-1m.csv.zip"
        try:
            import zipfile, io
            resp = requests.get(url, timeout=30, headers=HEADERS, stream=True)
            resp.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                text = z.read(z.namelist()[0]).decode("utf-8")
            cache_file.write_text(text, encoding="utf-8")
            log.info("Đã lưu cache Tranco → %s", cache_file)
        except Exception as e:
            log.error("Không tải được Tranco: %s", e)
            log.info("Dùng PRIORITY_DOMAINS thay thế")
            return PRIORITY_DOMAINS

    domains = []
    for line in text.strip().splitlines()[:top_n]:
        parts = line.strip().split(",")
        if len(parts) >= 2:
            domains.append(parts[1].strip().lower())

    # Đưa PRIORITY_DOMAINS lên đầu (đảm bảo chạy trước)
    priority_set = set(PRIORITY_DOMAINS)
    ordered = PRIORITY_DOMAINS + [d for d in domains if d not in priority_set]
    log.info("Tổng domain sẽ thử: %d (priority: %d)", len(ordered), len(PRIORITY_DOMAINS))
    return ordered


# ── Sitemap parser ────────────────────────────────────────────────────────────

def parse_locs(text: str) -> list:
    return re.findall(r"<loc>\s*(https?://[^<\s]+)\s*</loc>", text)


def try_get(url: str) -> str | None:
    """GET request, trả về text nếu thành công, None nếu lỗi."""
    try:
        resp = requests.get(
            url, timeout=HTTP_TIMEOUT,
            headers=HEADERS, allow_redirects=True
        )
        if resp.status_code == 200 and "<loc>" in resp.text:
            return resp.text
    except requests.RequestException:
        pass
    return None


def fetch_sitemap(domain: str, max_urls: int = MAX_PER_DOMAIN) -> list:
    """
    Thử nhiều sitemap URL, xử lý sitemap index, trả về list URL hợp lệ.
    """
    sitemap_urls = [
        f"https://{domain}/sitemap.xml",
        f"https://{domain}/sitemap_index.xml",
        f"https://www.{domain}/sitemap.xml",
        f"https://{domain}/sitemap/sitemap.xml",
        f"https://{domain}/post-sitemap.xml",
        f"https://{domain}/page-sitemap.xml",
        f"https://{domain}/news-sitemap.xml",
        f"https://{domain}/article-sitemap.xml",
    ]

    all_urls = []

    for sm_url in sitemap_urls:
        if len(all_urls) >= max_urls * 5:
            break

        text = try_get(sm_url)
        if not text:
            continue

        # Sitemap index → thử 3 sub-sitemap đầu
        if "<sitemapindex" in text:
            sub_locs = re.findall(
                r"<sitemap>.*?<loc>\s*(https?://[^<\s]+)\s*</loc>",
                text, re.DOTALL
            )
            for sub_loc in sub_locs[:3]:
                sub_text = try_get(sub_loc)
                if sub_text:
                    all_urls.extend(parse_locs(sub_text))
                time.sleep(0.2)
        else:
            all_urls.extend(parse_locs(text))

    # Filter + deduplicate
    valid = list(dict.fromkeys(
        u for u in all_urls
        if is_valid_url(u) and netloc_matches(domain_of(u), domain)
    ))

    return random.sample(valid, min(max_urls, len(valid))) if valid else []


# ── Main collection loop ──────────────────────────────────────────────────────

def collect(domains: list) -> list:
    all_urls = []
    seen = set()
    domain_got: dict = defaultdict(int)

    bar = tqdm(domains, desc="Sitemap")

    for domain in bar:
        if len(all_urls) >= TARGET_TOTAL:
            log.info("Đạt target %d URLs, dừng sớm.", TARGET_TOTAL)
            break

        bar.set_postfix(total=len(all_urls), domain=domain[:20])

        urls = fetch_sitemap(domain, MAX_PER_DOMAIN)
        new_urls = [u for u in urls if u not in seen]

        if new_urls:
            seen.update(new_urls)
            all_urls.extend(new_urls)
            domain_got[domain] = len(new_urls)
            log.info("  ✓ %-35s → %3d URLs", domain, len(new_urls))
        else:
            log.debug("  ✗ %-35s → 0", domain)

        time.sleep(SLEEP_OK)

    return all_urls


# ── Post-processing + save ────────────────────────────────────────────────────

def cap_per_domain(urls: list, cap: int = MAX_PER_DOMAIN) -> list:
    counts: dict = defaultdict(int)
    result = []
    for u in urls:
        d = domain_of(u)
        if counts[d] < cap:
            result.append(u)
            counts[d] += 1
    return result


def save_csv(urls: list, filepath: str) -> None:
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["url", "domain", "label"])
        for url in urls:
            writer.writerow([url, domain_of(url), "legit"])
    log.info("Đã lưu %d URLs → %s", len(urls), filepath)


def print_stats(urls: list) -> None:
    counts: dict = defaultdict(int)
    for u in urls:
        counts[domain_of(u)] += 1

    log.info("\n── Thống kê ──────────────────────────────────────────")
    log.info("  Tổng URLs : %d", len(urls))
    log.info("  Số domain : %d", len(counts))
    log.info("\n  Top 20 domain:")
    for d, cnt in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:20]:
        log.info("  %-38s %3d  %s", d, cnt, "█" * cnt)


# ── Entry point ───────────────────────────────────────────────────────────────

def main() -> None:
    log.info("🚀 Legit URL Collector — Phishing Detection Dataset")
    log.info("   Target: %d | Max/domain: %d\n", TARGET_TOTAL, MAX_PER_DOMAIN)

    # Load domain list
    domains = load_tranco_domains(TRANCO_TOP_N)

    # Collect
    all_urls = collect(domains)

    # Post-process
    log.info("\n── Post-processing ───────────────────────────────────")
    before = len(all_urls)
    # deduplicate đã làm trong collect(), chỉ cần cap
    all_urls = cap_per_domain(all_urls, MAX_PER_DOMAIN)
    log.info("  Cap/domain (%d): %d → %d", MAX_PER_DOMAIN, before, len(all_urls))

    random.shuffle(all_urls)
    save_csv(all_urls, OUTPUT_FILE)
    print_stats(all_urls)

    log.info("\n✅ Xong! File: %s | Log: %s", OUTPUT_FILE, LOG_FILE)


if __name__ == "__main__":
    main()

Sitemap: 100%|██████████| 723/723 [2:14:17<00:00, 11.14s/it, domain=justice.gov, total=7136]


In [ ]:
"""
collect_legit_urls_part2.py
=====================
Thu thập tiếp Legit URLs từ hạng 701 đến 3000 trong danh sách Tranco.
"""

import csv
import logging
import random
import re
import time
from collections import defaultdict
from io import StringIO
from pathlib import Path
from urllib.parse import urlparse
import csv
import requests
from tqdm import tqdm

# ── Cấu hình ────────────────────────────────────────────────────────────────

# ĐỔI TÊN FILE để không ghi đè mất 7k dữ liệu cũ của bạn
OUTPUT_FILE    = "/content/drive/MyDrive/Đồ án tốt nghiệp/legit_urls.csv"
LOG_FILE       = "/content/drive/MyDrive/Đồ án tốt nghiệp/collect.log"
TARGET_TOTAL   = 222319  # Mục tiêu đợt này (để cộng với 7k cũ là hơn 20k)
MAX_PER_DOMAIN = 50
HTTP_TIMEOUT   = 10
SLEEP_OK       = 0.5
SLEEP_ERR      = 1.0

# Vị trí bắt đầu và kết thúc lấy trong Tranco
TRANCO_START   = 1
TRANCO_END     = 300000

SKIP_EXT = {
    ".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp", ".ico",
    ".pdf", ".zip", ".rar", ".gz", ".tar",
    ".xml", ".css", ".js", ".json",
    ".mp4", ".mp3", ".avi", ".mov",
    ".woff", ".woff2", ".ttf", ".eot",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# Để rỗng để tránh crawl lại danh sách ưu tiên của lần chạy trước
PRIORITY_DOMAINS = []

# ── Logging ──────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)

# ── Helpers ──────────────────────────────────────────────────────────────────

def is_valid_url(url: str) -> bool:
    try:
        p = urlparse(url.strip())
    except Exception:
        return False
    if p.scheme not in ("http", "https"):
        return False
    if not p.netloc:
        return False
    if not p.path or p.path == "/":
        return False
    if Path(p.path).suffix.lower() in SKIP_EXT:
        return False
    return True


def domain_of(url: str) -> str:
    return urlparse(url).netloc.lower()


def netloc_matches(netloc: str, domain: str) -> bool:
    return netloc == domain or netloc.endswith("." + domain)


# ── Tranco list ───────────────────────────────────────────────────────────────

def load_tranco_domains(start_idx: int = TRANCO_START, end_idx: int = TRANCO_END) -> list:
    """
    Download Tranco top-1M list (file CSV, ~10MB).
    Lấy từ vị trí start_idx đến end_idx.
    """
    cache_file = Path(".tranco_cache.csv")

    if cache_file.exists():
        log.info("Dùng Tranco cache: %s", cache_file)
        text = cache_file.read_text(encoding="utf-8")
    else:
        log.info("Đang download Tranco list...")
        url = "https://tranco-list.eu/top-1m.csv.zip"
        try:
            import zipfile, io
            resp = requests.get(url, timeout=30, headers=HEADERS, stream=True)
            resp.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                text = z.read(z.namelist()[0]).decode("utf-8")
            cache_file.write_text(text, encoding="utf-8")
            log.info("Đã lưu cache Tranco → %s", cache_file)
        except Exception as e:
            log.error("Không tải được Tranco: %s", e)
            return []

    domains = []
    # CẮT MẢNG TỪ START ĐẾN END
    for line in text.strip().splitlines()[start_idx:end_idx]:
        parts = line.strip().split(",")
        if len(parts) >= 2:
            domains.append(parts[1].strip().lower())

    ordered = PRIORITY_DOMAINS + [d for d in domains if d not in set(PRIORITY_DOMAINS)]
    log.info("Tổng domain sẽ thử: %d (Từ rank %d đến %d)", len(ordered), start_idx + 1, end_idx)
    return ordered


# ── Sitemap parser ────────────────────────────────────────────────────────────

def parse_locs(text: str) -> list:
    return re.findall(r"<loc>\s*(https?://[^<\s]+)\s*</loc>", text)


def try_get(url: str) -> str | None:
    try:
        resp = requests.get(
            url, timeout=HTTP_TIMEOUT,
            headers=HEADERS, allow_redirects=True
        )
        if resp.status_code == 200 and "<loc>" in resp.text:
            return resp.text
    except requests.RequestException:
        pass
    return None


def fetch_sitemap(domain: str, max_urls: int = MAX_PER_DOMAIN) -> list:
    sitemap_urls = [
        f"https://{domain}/sitemap.xml",
        f"https://{domain}/sitemap_index.xml",
        f"https://www.{domain}/sitemap.xml",
        f"https://{domain}/sitemap/sitemap.xml",
        f"https://{domain}/post-sitemap.xml",
        f"https://{domain}/page-sitemap.xml",
        f"https://{domain}/news-sitemap.xml",
        f"https://{domain}/article-sitemap.xml",
    ]

    all_urls = []

    for sm_url in sitemap_urls:
        if len(all_urls) >= max_urls * 5:
            break

        text = try_get(sm_url)
        if not text:
            continue

        if "<sitemapindex" in text:
            sub_locs = re.findall(
                r"<sitemap>.*?<loc>\s*(https?://[^<\s]+)\s*</loc>",
                text, re.DOTALL
            )
            for sub_loc in sub_locs[:3]:
                sub_text = try_get(sub_loc)
                if sub_text:
                    all_urls.extend(parse_locs(sub_text))
                time.sleep(0.2)
        else:
            all_urls.extend(parse_locs(text))

    valid = list(dict.fromkeys(
        u for u in all_urls
        if is_valid_url(u) and netloc_matches(domain_of(u), domain)
    ))

    return random.sample(valid, min(max_urls, len(valid))) if valid else []


# ── Main collection loop ──────────────────────────────────────────────────────

def collect(domains: list) -> list:
    all_urls = []
    seen = set()
    domain_got: dict = defaultdict(int)

    bar = tqdm(domains, desc="Sitemap")

    for domain in bar:
        if len(all_urls) >= TARGET_TOTAL:
            log.info("Đạt target %d URLs, dừng sớm.", TARGET_TOTAL)
            break

        bar.set_postfix(total=len(all_urls), domain=domain[:20])

        urls = fetch_sitemap(domain, MAX_PER_DOMAIN)
        new_urls = [u for u in urls if u not in seen]

        if new_urls:
            seen.update(new_urls)
            all_urls.extend(new_urls)
            domain_got[domain] = len(new_urls)
            log.info("  ✓ %-35s → %3d URLs", domain, len(new_urls))
        else:
            log.debug("  ✗ %-35s → 0", domain)

        time.sleep(SLEEP_OK)

    return all_urls


# ── Post-processing + save ────────────────────────────────────────────────────

def cap_per_domain(urls: list, cap: int = MAX_PER_DOMAIN) -> list:
    counts: dict = defaultdict(int)
    result = []
    for u in urls:
        d = domain_of(u)
        if counts[d] < cap:
            result.append(u)
            counts[d] += 1
    return result


def save_csv(urls: list, filepath: str) -> None:
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["url", "domain", "label"])
        for url in urls:
            writer.writerow([url, domain_of(url), "legit"])
    log.info("Đã lưu %d URLs → %s", len(urls), filepath)


def print_stats(urls: list) -> None:
    counts: dict = defaultdict(int)
    for u in urls:
        counts[domain_of(u)] += 1

    log.info("\n── Thống kê ──────────────────────────────────────────")
    log.info("  Tổng URLs : %d", len(urls))
    log.info("  Số domain : %d", len(counts))
    log.info("\n  Top 20 domain:")
    for d, cnt in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:20]:
        log.info("  %-38s %3d  %s", d, cnt, "█" * cnt)


# ── Entry point ───────────────────────────────────────────────────────────────

def main() -> None:
    log.info("🚀 Legit URL Collector — Chạy Part 2")
    log.info("   Target: %d | Max/domain: %d\n", TARGET_TOTAL, MAX_PER_DOMAIN)

    domains = load_tranco_domains()
    all_urls = collect(domains)

    log.info("\n── Post-processing ───────────────────────────────────")
    before = len(all_urls)
    all_urls = cap_per_domain(all_urls, MAX_PER_DOMAIN)
    log.info("  Cap/domain (%d): %d → %d", MAX_PER_DOMAIN, before, len(all_urls))

    random.shuffle(all_urls)
    save_csv(all_urls, OUTPUT_FILE)
    print_stats(all_urls)

    log.info("\n✅ Xong! File: %s | Log: %s", OUTPUT_FILE, LOG_FILE)


if __name__ == "__main__":
    main()

Sitemap:  57%|█████▋    | 1305/2300 [3:43:24<2:50:19, 10.27s/it, domain=mindbox.ru, total=14981]


# Legitimate

In [ ]:
"""
collect_legit_urls_async.py
============================
Phiên bản ASYNC — nhanh hơn ~20-50x so với bản tuần tự.
Dùng aiohttp + asyncio để crawl nhiều domain cùng lúc.

Cài thêm: pip install aiohttp tqdm
"""

import asyncio
import csv
import logging
import random
import re
import time
from collections import defaultdict
from pathlib import Path
from urllib.parse import urlparse

import aiohttp
from tqdm.asyncio import tqdm

# ── Cấu hình ────────────────────────────────────────────────────────────────

OUTPUT_FILE    = "/content/drive/MyDrive/Đồ án tốt nghiệp/legit_urls.csv"
LOG_FILE       = "/content/drive/MyDrive/Đồ án tốt nghiệp/collect.log"
TARGET_TOTAL   = 222319
MAX_PER_DOMAIN = 50
HTTP_TIMEOUT   = 10          # giây
SLEEP_OK       = 0.0         # async không cần sleep giữa các domain
TRANCO_START   = 1
TRANCO_END     = 300000

# ── Điều chỉnh độ song song ──────────────────────────────────────────────────
# Tăng lên nếu mạng tốt, giảm nếu bị rate-limit hoặc lỗi nhiều
CONCURRENT_DOMAINS   = 100    # số domain xử lý đồng thời
CONCURRENT_SITEMAPS  = 6     # số sitemap con fetch song song mỗi domain

SKIP_EXT = {
    ".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp", ".ico",
    ".pdf", ".zip", ".rar", ".gz", ".tar",
    ".xml", ".css", ".js", ".json",
    ".mp4", ".mp3", ".avi", ".mov",
    ".woff", ".woff2", ".ttf", ".eot",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# ── Logging ──────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)

# ── Helpers ──────────────────────────────────────────────────────────────────

def is_valid_url(url: str) -> bool:
    try:
        p = urlparse(url.strip())
    except Exception:
        return False
    if p.scheme not in ("http", "https"):
        return False
    if not p.netloc:
        return False
    if not p.path or p.path == "/":
        return False
    if Path(p.path).suffix.lower() in SKIP_EXT:
        return False
    return True


def domain_of(url: str) -> str:
    return urlparse(url).netloc.lower()


def netloc_matches(netloc: str, domain: str) -> bool:
    return netloc == domain or netloc.endswith("." + domain)


def parse_locs(text: str) -> list[str]:
    return re.findall(r"<loc>\s*(https?://[^<\s]+)\s*</loc>", text)


# ── Tranco list ───────────────────────────────────────────────────────────────

def load_tranco_domains() -> list[str]:
    cache_file = Path(".tranco_cache.csv")
    if cache_file.exists():
        log.info("Dùng Tranco cache: %s", cache_file)
        text = cache_file.read_text(encoding="utf-8")
    else:
        log.info("Đang download Tranco list...")
        import zipfile, io, requests
        url = "https://tranco-list.eu/top-1m.csv.zip"
        try:
            resp = requests.get(url, timeout=60, headers=HEADERS, stream=True)
            resp.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                text = z.read(z.namelist()[0]).decode("utf-8")
            cache_file.write_text(text, encoding="utf-8")
            log.info("Đã lưu cache Tranco → %s", cache_file)
        except Exception as e:
            log.error("Không tải được Tranco: %s", e)
            return []

    domains = []
    for line in text.strip().splitlines()[TRANCO_START:TRANCO_END]:
        parts = line.strip().split(",")
        if len(parts) >= 2:
            domains.append(parts[1].strip().lower())

    log.info("Tổng domain sẽ thử: %d (rank %d → %d)", len(domains), TRANCO_START + 1, TRANCO_END)
    return domains


# ── Async sitemap fetcher ─────────────────────────────────────────────────────

async def async_get(session: aiohttp.ClientSession, url: str) -> str | None:
    """Fetch 1 URL, trả về text nếu có <loc>, None nếu lỗi hoặc không phải sitemap."""
    try:
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=HTTP_TIMEOUT)) as resp:
            if resp.status == 200:
                text = await resp.text(errors="replace")
                if "<loc>" in text:
                    return text
    except Exception:
        pass
    return None


async def fetch_sitemap_async(
    session: aiohttp.ClientSession,
    domain: str,
    max_urls: int = MAX_PER_DOMAIN,
) -> list[str]:
    sitemap_candidates = [
        f"https://{domain}/sitemap.xml",
        f"https://{domain}/sitemap_index.xml",
        f"https://www.{domain}/sitemap.xml",
        f"https://{domain}/sitemap/sitemap.xml",
        f"https://{domain}/post-sitemap.xml",
        f"https://{domain}/page-sitemap.xml",
        f"https://{domain}/news-sitemap.xml",
        f"https://{domain}/article-sitemap.xml",
    ]

    all_urls: list[str] = []

    for sm_url in sitemap_candidates:
        if len(all_urls) >= max_urls * 5:
            break

        text = await async_get(session, sm_url)
        if not text:
            continue

        if "<sitemapindex" in text:
            # Fetch sub-sitemaps song song (giới hạn CONCURRENT_SITEMAPS)
            sub_locs = re.findall(
                r"<sitemap>.*?<loc>\s*(https?://[^<\s]+)\s*</loc>",
                text, re.DOTALL
            )[:5]
            tasks = [async_get(session, loc) for loc in sub_locs[:CONCURRENT_SITEMAPS]]
            results = await asyncio.gather(*tasks)
            for sub_text in results:
                if sub_text:
                    all_urls.extend(parse_locs(sub_text))
        else:
            all_urls.extend(parse_locs(text))

    valid = list(dict.fromkeys(
        u for u in all_urls
        if is_valid_url(u) and netloc_matches(domain_of(u), domain)
    ))

    return random.sample(valid, min(max_urls, len(valid))) if valid else []


# ── Worker pool ───────────────────────────────────────────────────────────────

async def collect_async(domains: list[str]) -> list[str]:
    """
    Chạy CONCURRENT_DOMAINS domain song song.
    Dừng khi đủ TARGET_TOTAL URLs.
    """
    all_urls: list[str] = []
    seen: set[str] = set()
    done_count = 0
    semaphore = asyncio.Semaphore(CONCURRENT_DOMAINS)

    # Connector: giới hạn số connection mở cùng lúc
    connector = aiohttp.TCPConnector(
        limit=CONCURRENT_DOMAINS * 4,
        ttl_dns_cache=300,
        ssl=False,                  # tắt verify SSL để nhanh hơn, bỏ nếu cần an toàn
    )

    async def process_domain(session: aiohttp.ClientSession, domain: str, pbar) -> None:
        nonlocal done_count
        async with semaphore:
            if len(all_urls) >= TARGET_TOTAL:
                return
            urls = await fetch_sitemap_async(session, domain, MAX_PER_DOMAIN)
            new_urls = [u for u in urls if u not in seen]
            if new_urls:
                seen.update(new_urls)
                all_urls.extend(new_urls)
                log.info("  ✓ %-35s → %3d URLs  (tổng: %d)", domain, len(new_urls), len(all_urls))
            done_count += 1
            pbar.set_postfix(total=len(all_urls), done=done_count)
            pbar.update(1)

    async with aiohttp.ClientSession(headers=HEADERS) as session:
        with tqdm(total=len(domains), desc="Async sitemap") as pbar:
            tasks = [process_domain(session, d, pbar) for d in domains]
            # asyncio.gather chạy tất cả, semaphore giới hạn CONCURRENT_DOMAINS chạy cùng lúc
            await asyncio.gather(*tasks)

    return all_urls


# ── Post-processing + save ────────────────────────────────────────────────────

def cap_per_domain(urls: list[str], cap: int = MAX_PER_DOMAIN) -> list[str]:
    counts: dict[str, int] = defaultdict(int)
    result = []
    for u in urls:
        d = domain_of(u)
        if counts[d] < cap:
            result.append(u)
            counts[d] += 1
    return result


def save_csv(urls: list[str], filepath: str) -> None:
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["url", "domain", "label"])
        for url in urls:
            writer.writerow([url, domain_of(url), "legit"])
    log.info("Đã lưu %d URLs → %s", len(urls), filepath)


def print_stats(urls: list[str]) -> None:
    counts: dict[str, int] = defaultdict(int)
    for u in urls:
        counts[domain_of(u)] += 1
    log.info("\n── Thống kê ──────────────────────────────────────────")
    log.info("  Tổng URLs : %d", len(urls))
    log.info("  Số domain : %d", len(counts))
    log.info("\n  Top 20 domain:")
    for d, cnt in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:20]:
        log.info("  %-38s %3d", d, cnt)


# ── Entry point ───────────────────────────────────────────────────────────────

def main() -> None:
    t0 = time.time()
    log.info("🚀 Legit URL Collector — ASYNC version")
    log.info("   Target: %d | Max/domain: %d | Concurrent: %d\n",
             TARGET_TOTAL, MAX_PER_DOMAIN, CONCURRENT_DOMAINS)

    domains = load_tranco_domains()

    # asyncio.run() cho script thường; trong Colab/Jupyter dùng await collect_async(domains)
    try:
        all_urls = asyncio.run(collect_async(domains))
    except RuntimeError:
        # Đang chạy trong Jupyter/Colab (event loop đã tồn tại)
        import nest_asyncio
        nest_asyncio.apply()
        loop = asyncio.get_event_loop()
        all_urls = loop.run_until_complete(collect_async(domains))

    log.info("\n── Post-processing ───────────────────────────────────")
    before = len(all_urls)
    all_urls = cap_per_domain(all_urls, MAX_PER_DOMAIN)
    log.info("  Cap/domain (%d): %d → %d", MAX_PER_DOMAIN, before, len(all_urls))

    random.shuffle(all_urls)
    save_csv(all_urls, OUTPUT_FILE)
    print_stats(all_urls)

    elapsed = time.time() - t0
    log.info("\n✅ Xong! Thời gian: %.1f giây (%.1f phút)", elapsed, elapsed / 60)
    log.info("   File: %s | Log: %s", OUTPUT_FILE, LOG_FILE)


if __name__ == "__main__":
    main()

Async sitemap:   6%|▌         | 17956/299999 [36:22<9:31:21,  8.23it/s, done=17956, total=223031]  


# Phishing

In [2]:
import requests
import pandas as pd
import time
import random
from tqdm import tqdm

def crawl_phishing_optimized(target_count=260000, size_per_page=100):
    all_data = []
    seen_ids = set()
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    # 1. Dùng Session để giữ TCP connection ổn định, ít bị WAF block hơn
    session = requests.Session()
    session.headers.update(headers)

    print(f"🚀 Bắt đầu crawl {target_count:,} URL...")

    pbar = tqdm(total=target_count, desc="Crawling", unit=" records")
    consecutive_empty = 0
    last_id = None  # Mốc để phân trang theo ID

    while len(all_data) < target_count:
        # 2. Phân trang bằng ID (Tránh Deep Pagination làm nặng server)
        if last_id is None:
            url = f"https://api.phishstats.info/api/phishing?_size={size_per_page}&_sort=-id"
        else:
            url = f"https://api.phishstats.info/api/phishing?_where=(id,lt,{last_id})&_size={size_per_page}&_sort=-id"

        success = False
        for attempt in range(5):
            try:
                res = session.get(url, timeout=15)

                if res.status_code == 429:
                    wait = 90 * (attempt + 1)
                    tqdm.write(f"⚠️ Rate limit (attempt {attempt+1}). Chờ {wait}s...")
                    time.sleep(wait)
                    continue

                res.raise_for_status()
                data = res.json()

                if not data:
                    consecutive_empty += 1
                    if consecutive_empty >= 3:
                        tqdm.write("⛔ Hết data từ API.")
                        success = True
                    break

                consecutive_empty = 0
                new_count = 0
                current_min_id = float('inf')

                for r in data:
                    rid = r.get("id")
                    if rid:
                        current_min_id = min(current_min_id, rid)
                        if rid not in seen_ids:
                            seen_ids.add(rid)
                            all_data.append(r)
                            new_count += 1

                if new_count > 0:
                    last_id = current_min_id  # Cập nhật ID nhỏ nhất để làm mốc cho vòng lặp sau
                else:
                    consecutive_empty += 1

                pbar.update(new_count)
                success = True

                # 3. Thêm độ nhiễu vào thời gian chờ (4.0s - 5.5s) để qua mặt Bot Protection
                time.sleep(random.uniform(4.0, 5.5))
                break

            except requests.exceptions.RequestException as e:
                tqdm.write(f"❌ Lỗi: {e}")
                time.sleep(15)

        if not success or consecutive_empty >= 3:
            tqdm.write("⏭️ Quá số lần thử hoặc không có data mới, kết thúc vòng lặp.")
            break

    pbar.close()

    df = pd.DataFrame(all_data)
    if df.empty or "url" not in df.columns:
        print("⚠️ Không có dữ liệu.")
        return

    before = len(df)
    df.sort_values("id", ascending=False, inplace=True)
    df.drop_duplicates(subset=["url"], keep="first", inplace=True)
    print(f"\n🧹 Dedup: {before:,} → {len(df):,} (bỏ {before-len(df):,})")

    df_ml = df.head(target_count)[["url"]].copy()
    df_ml["label"] = 0

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        print(f"🔥 Mới nhất : {df['date'].iloc[0]}")
        print(f"🕰️ Cũ nhất  : {df['date'].iloc[-1]}")
        print(f"📅 Trải dài : {(df['date'].max() - df['date'].min()).days} ngày")

    filename = "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/phishstats_ml_dataset.csv"
    df_ml.to_csv(filename, index=False, encoding="utf-8")
    print(f"✅ Đã lưu {len(df_ml):,} URL → {filename}")

if __name__ == "__main__":
    crawl_phishing_optimized(target_count=260000)

🚀 Bắt đầu crawl 260,000 URL...


Crawling: 100%|██████████| 260000/260000 [3:28:57<00:00, 20.74 records/s]



🧹 Dedup: 260,000 → 259,840 (bỏ 160)
🔥 Mới nhất : 2026-05-11 08:02:42+00:00
🕰️ Cũ nhất  : 2025-08-19 04:06:26+00:00
📅 Trải dài : 265 ngày
✅ Đã lưu 259,840 URL → /content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/phishstats_ml_dataset.csv


In [3]:
import pandas as pd

# 1. Đọc file CSV
# Lưu ý: Nhớ đóng dấu ngoặc kép ở cuối đường dẫn
df2 = pd.read_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_accessible.csv")

# 2. Loại bỏ các dòng có label = 0
# Cách 1: Chỉ giữ lại những dòng có label khác 0
df2 = df2[df2['label'] != 0]

# Cách 2 (Nếu bạn chỉ muốn lấy label 1):
# df2 = df2[df2['label'] == 1]

# 3. Kiểm tra lại kết quả
print("Các label còn lại trong df2:")
print(df2['label'].unique())
print(f"\nKích thước tập dữ liệu sau khi lọc: {df2.shape}")

Các label còn lại trong df2:
[1]

Kích thước tập dữ liệu sau khi lọc: (222963, 4)


In [4]:
df1 = pd.read_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/phishing_valid_html.csv")
# df2 = pd.read_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_accessible.csv)
df = pd.concat([df1, df2], ignore_index=True)
df.dropna()

,url,label,accessible,status_code
97797,https://torontosun.com/news/local-news/peel-co...,1,True,200.0
97798,https://www.androidcentral.com/author/danny-ze...,1,True,200.0
97799,https://www.reverbnation.com/collection/6629-d...,1,True,200.0
97800,https://www.recombee.com/faq,1,True,200.0
97801,https://www.speedtest.net/global-index/barbados,1,True,200.0
...,...,...,...,...
320755,https://www.cuhk.edu.hk/english/campus/cuhk-ca...,1,True,200.0
320756,https://www.proxibid.com/Loy-Real-Estate-Aucti...,1,True,200.0
320757,https://www.bbc.co.uk/news/topics/c8zwnrr1me9t,1,True,200.0
320758,https://www.5paisa.com/stocks/dcxindia-share-p...,1,True,200.0


In [6]:
df.to_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls.csv", index=False)